# Modern Optimizers: From Adam to Muon

> In 11-training-loss we wrote `optimizer = torch.optim.AdamW(model.parameters(), lr=...)`, then `optimizer.step()` nudged the parameters one step forward. What actually happens inside that step, and why AdamW instead of SGD, this appendix has not covered.
>
> This section starts from the two equations behind Adam and walks through AdamW, gradient clipping, Muon, Lion/Shampoo/SOAP, finishing with an empirical comparison on miniGPT. The goal is to build engineering intuition for "which problem each optimizer solves", not to pile up math.


An optimizer decides one thing: given a gradient, which direction the parameters move and how far. The most naive SGD updates directly with $\theta \leftarrow \theta - \eta g$, but in the LLM regime of billions of parameters and tens of thousands of steps, SGD barely converges — the gradient is noisy, the effective curvature differs sharply across parameter directions, and the early-step update direction is unstable. Adam introduces exponential moving averages of the first and second moments, effectively computing a separate "adaptive learning rate" for each parameter, and has become the de facto standard for LLM training over the past decade.

Adam is not the endpoint. In 2024 Keller Jordan used Muon in the nanoGPT Speedrun to cut training time in half; second-order optimizers (Shampoo, SOAP) re-entered the picture; Lion showed memory advantages over AdamW on large models. The shared idea behind these new methods is to treat gradients as matrices and exploit matrix structure (orthogonalization, low-rank approximation) to produce finer update directions than "divide element-wise by the second moment". To follow this line of evolution, we first need to get the engineering details of Adam itself straight.

This section assumes the reader has read 11-training-loss (knows the relationship between loss, backward, and optimizer.step()) and 06-mini-gpt (knows the MiniGPT architecture). We will first implement AdamW by hand, then a simplified Muon, and finally run a real loss curve comparison on miniGPT.


## 1. Adam: First Moment + Second Moment

Adam maintains two states per parameter tensor: the first moment $m$ (the moving average of the gradient, also called momentum) and the second moment $v$ (the moving average of squared gradients, an approximation of the variance). At step $t$, after obtaining gradient $g_t$, the update rules are:

$$m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t$$
$$v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2$$

The role of $m$ is "direction accumulation": if the gradient points the same way for several consecutive steps, $m$ grows larger, weighting that direction more; if the gradient oscillates back and forth, positives and negatives cancel, and $m$ approaches 0. This keeps progress stable even under noisy gradients. The role of $v$ is "element-wise normalization": each parameter position has its own $v$, and at update time the denominator is $\sqrt{v} + \epsilon$, which suppresses the learning rate for positions whose long-term gradient is large — this is where "adaptive learning rate" comes from.

Both moments are initialized to 0, so the estimates in the first few steps are biased toward 0 (both $m$ and $v$ are too small). Adam fixes this with bias correction:

$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$

When $t$ is large, $\beta^t \to 0$ and the correction term approaches 1, equivalent to no correction. It only magnifies the estimate significantly early on — without this step, the model's updates in the first few thousand steps would be abnormally slow, wasting the warmup period. The final update equation:

$$\theta_t = \theta_{t-1} - \eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

Typical hyperparameters: $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\epsilon = 10^{-8}$, and $\eta$ in LLM pretraining is usually on the order of $10^{-4}$. $\beta_2$ is set to 0.999 because the second moment of the gradient changes more slowly than the first moment, requiring a longer moving window for a stable estimate.


### 1.1 Hand-calculating one Adam step

Suppose a parameter has initial value $\theta_0 = 1.0$, the first-step gradient is $g_1 = 0.5$, and the hyperparameters are $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\eta = 0.1$, $\epsilon = 10^{-8}$. Let $m_0 = v_0 = 0$.

- $m_1 = 0.9 \cdot 0 + 0.1 \cdot 0.5 = 0.05$
- $v_1 = 0.999 \cdot 0 + 0.001 \cdot 0.25 = 0.00025$
- $\hat{m}_1 = 0.05 / (1 - 0.9^1) = 0.05 / 0.1 = 0.5$ (equal to $g_1$; bias correction fully restores the single-step estimate)
- $\hat{v}_1 = 0.00025 / (1 - 0.999^1) = 0.00025 / 0.001 = 0.25$ (equal to $g_1^2$)
- $\Delta\theta = -0.1 \cdot 0.5 / (\sqrt{0.25} + 10^{-8}) = -0.1 \cdot 0.5 / 0.5 = -0.1$
- $\theta_1 = 1.0 - 0.1 = 0.9$

Key observation: bias correction makes the effective gradient of the first step exactly equal to $g_1$ itself; otherwise $m_1 = 0.05$ would be too small and $v_1 = 0.00025$ would make the denominator tiny, severely underestimating the update magnitude.


In [ ]:
import torch

torch.manual_seed(42)

# Hand-check: a single parameter takes one Adam step
theta = torch.tensor([1.0], requires_grad=True)
theta.grad = torch.tensor([0.5])

optimizer = torch.optim.Adam([theta], lr=0.1, betas=(0.9, 0.999), eps=1e-8)
optimizer.step()

print(f"theta before update: 1.0")
print(f"theta after update:  {theta.item():.6f}")
print(f"hand-computed expect: 0.9")
print(f"error:               {abs(theta.item() - 0.9):.2e}")
print()
print("Key observation: bias correction at step 1 makes the Adam update magnitude equal to lr * sign(grad).")
print("Without correction, the magnitude would be severely underestimated (m=0.05, v=0.00025 are too small).")


### 1.2 Implementing a simplified Adam

The fastest way to understand Adam is to write a minimal implementation that runs. Subclass `torch.optim.Optimizer` and update parameters according to the formula in `step()`. This code matches the core logic of the official PyTorch implementation, omitting only engineering optimizations such as AMP and foreach.


In [ ]:
import torch


class SimpleAdam(torch.optim.Optimizer):
    """Simplified Adam retaining only the core logic, easy to read against the formulas."""

    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        defaults = {"lr": lr, "betas": betas, "eps": eps}
        super().__init__(params, defaults)

    def step(self):
        for group in self.param_groups:
            lr = group["lr"]
            beta1, beta2 = group["betas"]
            eps = group["eps"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                grad = p.grad.data
                state = self.state[p]

                # Initialize state on first access
                if "t" not in state:
                    state["t"] = 0
                    state["m"] = torch.zeros_like(p.data)
                    state["v"] = torch.zeros_like(p.data)

                state["t"] += 1
                t = state["t"]
                m, v = state["m"], state["v"]

                # Exponential moving average of first and second moments
                m.mul_(beta1).add_(grad, alpha=1 - beta1)
                v.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)

                # bias correction
                m_hat = m / (1 - beta1 ** t)
                v_hat = v / (1 - beta2 ** t)

                # parameter update
                p.data.sub_(lr * m_hat / (v_hat.sqrt() + eps))


# Verify: output is identical to the official PyTorch Adam under the same inputs
torch.manual_seed(0)
p1 = torch.randn(100, requires_grad=True)
p2 = p1.clone().detach().requires_grad_(True)
grad = torch.randn(100) * 0.1
p1.grad = grad.clone()
p2.grad = grad.clone()

opt_torch = torch.optim.Adam([p1], lr=1e-3)
opt_ours = SimpleAdam([p2], lr=1e-3)
opt_torch.step()
opt_ours.step()

max_diff = (p1 - p2).abs().max().item()
print(f"PyTorch Adam vs SimpleAdam, max diff at step 1: {max_diff:.2e}")

# Take 99 more steps to check long-term consistency
for _ in range(99):
    g = torch.randn(100) * 0.1
    p1.grad = g.clone()
    p2.grad = g.clone()
    opt_torch.step()
    opt_ours.step()

max_diff = (p1 - p2).abs().max().item()
print(f"Max diff after 100 steps: {max_diff:.2e}")
print("Key observation: the hand-written implementation matches the official Adam numerically, so the formula is correct.")


## 2. AdamW: Decoupling Weight Decay

In the original Adam paper, if you want to add weight decay (L2 regularization), the recipe is to add $\lambda \theta$ to the gradient: $g_t^{\text{reg}} = g_t + \lambda \theta$, then feed this modified gradient into the updates of $m$ and $v$. This is equivalent to adding an L2 penalty of $\frac{\lambda}{2} \|\theta\|^2$ to the loss.

The problem is that the gradient of the L2 penalty, $\lambda \theta$, passes through the $m / \sqrt{v}$ normalization — that is, the strength of weight decay gets scaled by the adaptive learning rate. For parameters with a large historical gradient, $\sqrt{v}$ is large and the actual effect of weight decay is weakened; for parameters with a small historical gradient, weight decay is amplified instead. This breaks weight decay's intent of "pulling parameters toward 0 uniformly".

AdamW (Loshchilov & Hutter, 2019) decouples weight decay from the gradient chain: decay the parameters directly with $\theta \leftarrow \theta - \eta \lambda \theta$, bypassing $m / \sqrt{v}$. The update equation becomes:

$$\theta_t = \theta_{t-1} - \eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon} - \eta \lambda \theta_{t-1}$$

In practice the decay term is usually applied before the Adam update (`p.data -= lr * weight_decay * p.data`), so that the subsequent $\hat{m}/\sqrt{\hat{v}}$ is computed against the "decayed" parameters, which is more stable.


### 2.1 Which parameters to decay, which not to

The standard recipe: bias and LayerNorm parameters get no weight decay, everything else (Linear weights, Embedding) gets decay. The reason is that bias and LayerNorm "shift" the mean of the activations, so pulling them toward 0 is pointless or even harms expressiveness. Linear weights and Embeddings are "scaling"-type parameters, where weight decay prevents them from growing too large and saturating activations.

In PyTorch we distinguish via parameter groups:

```python
decay_params = [p for n, p in model.named_parameters() if p.requires_grad and p.dim() >= 2]
no_decay_params = [p for n, p in model.named_parameters() if p.requires_grad and p.dim() < 2]
optimizer = torch.optim.AdamW([
    {"params": decay_params, "weight_decay": 0.1},
    {"params": no_decay_params, "weight_decay": 0.0},
], lr=3e-4)
```

`p.dim() >= 2` is a common heuristic: tensors of 2D and above are usually Linear/Conv/Embedding weights, while 1D tensors are usually bias and LayerNorm $\gamma, \beta$. This test is not strict (some implementations match by name explicitly) but works well in practice.


In [ ]:
import torch
import torch.nn as nn


# A minimal decoder-only model skeleton, used to demonstrate parameter grouping
class TinyBlock(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.ln = nn.LayerNorm(d)
        self.fc = nn.Linear(d, d * 4)
        self.proj = nn.Linear(d * 4, d)

    def forward(self, x):
        return self.proj(torch.nn.functional.gelu(self.fc(self.ln(x))))


model = nn.Sequential(
    nn.Embedding(1000, 64),
    TinyBlock(64),
    TinyBlock(64),
    nn.Linear(64, 1000),
)

# Group by dimension: 2D and above decay, 1D does not
decay, no_decay = [], []
for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if p.dim() >= 2:
        decay.append(p)
    else:
        no_decay.append(p)

print(f"decay params:    {len(decay)}, total elements: {sum(p.numel() for p in decay)}")
print(f"no_decay params: {len(no_decay)}, total elements: {sum(p.numel() for p in no_decay)}")
print()
print("Typical split: decay is the vast majority (Linear/Embedding weights); no_decay is only bias and LayerNorm.")


## 3. Gradient Clipping

At the start of training, the model parameters are far from the optimum and the curvature of the loss surface is unpredictable. A single outlier batch (especially dirty data, especially long sequences) can produce a huge gradient that flings the parameters off beyond recovery. Gradient clipping caps the gradient norm to prevent any single-step update from being "catastrophically" large.

The most common form in LLMs is global norm clip: first compute the global L2 norm of gradients across all parameters $\|g\| = \sqrt{\sum_i \|g_i\|^2}$, and if it exceeds the threshold $\tau$, scale all gradients proportionally by $\tau / \|g\|$:

$$g \leftarrow g \cdot \min\left(1, \frac{\tau}{\|g\|}\right)$$

This preserves the relative direction of each parameter's gradient and only scales the overall magnitude. The threshold $\tau$ is typically set to 1.0 in large models such as GPT-3 and LLaMA. Another form, value clip, clips element-wise to $[-c, c]$, but it distorts the gradient direction and is rarely used in LLMs.

Why does the Transformer in particular need clipping? Early on, the softmax in attention has not converged yet, and a small number of anomalous activations can make some heads' gradients orders of magnitude larger than the average; residual connections then amplify this anomaly layer by layer. Without clipping, a single bad batch can blow up an entire layer's parameters.


In [ ]:
import torch


def clip_grad_norm_(params, max_norm, eps=1e-6):
    """Hand-written global norm clip, equivalent to torch.nn.utils.clip_grad_norm_

    Args:
        params: iterable of parameters (grad must not be None)
        max_norm: upper bound on the gradient global norm
        eps: small constant to avoid division by zero
    """
    # Collect all non-None gradients
    grads = [p.grad for p in params if p.grad is not None]
    if len(grads) == 0:
        return torch.tensor(0.0)

    # Compute global norm: square root of the sum of squared L2 norms of all gradients
    total_norm_sq = sum(g.float().pow(2).sum() for g in grads)
    total_norm = total_norm_sq.sqrt()

    # If it exceeds the threshold, scale all gradients proportionally
    clip_coef = max_norm / (total_norm + eps)
    if clip_coef < 1:
        for g in grads:
            g.mul_(clip_coef)

    return total_norm


# Verify: construct a set of gradients whose global norm far exceeds 1.0
params = [torch.randn(100, requires_grad=True) for _ in range(3)]
for p in params:
    p.grad = torch.randn(100) * 10  # deliberately magnify the gradient

before_norm = torch.sqrt(sum(p.grad.float().pow(2).sum() for p in params)).item()
print(f"global norm before clip: {before_norm:.2f}")

returned = clip_grad_norm_(params, max_norm=1.0)
after_norm = torch.sqrt(sum(p.grad.float().pow(2).sum() for p in params)).item()
print(f"returned (original) norm: {returned.item():.2f}")
print(f"global norm after clip: {after_norm:.4f}")
print()
print("Key observation: after clipping the global norm is pressed down near 1.0, but the gradient direction is preserved.")
print("Clipping never magnifies the gradient — when the original norm is below the threshold, clip_coef >= 1 but is not applied (here via `if clip_coef < 1`).")


## 4. Muon: Matrix-orthogonalized Gradient Updates

The key observation behind Muon (Keller Jordan, 2024) is that the weight of a Linear layer is a 2D matrix $W \in \mathbb{R}^{m \times n}$, and so is its gradient $G$. Adam flattens $G$ into a vector and applies $m/\sqrt{v}$ element-wise, discarding the matrix structure. If we treat $G$ as a matrix, we can orthogonalize it with a Newton-Schulz iteration and obtain an update with "a more stable direction and a more controllable step size".

The intuition for orthogonalization: take the SVD of the gradient $G = U \Sigma V^\top$, and orthogonalization replaces all singular values with 1, yielding $U V^\top$. The "magnitude" of the update direction is normalized, and only the "direction" is preserved. Intuitively this resembles sign-gradient, but it preserves all of the matrix's directional information.

Doing SVD directly is too slow. The Newton-Schulz iteration is an approximate orthogonalization algorithm:

$$X_{k+1} = \frac{1}{2} X_k (3 I - X_k^\top X_k)$$

Starting from $X_0 = G / \|G\|_F$, 7-10 iterations suffice to bring $X_k$ close to the orthogonalized version $U V^\top$ of $G$ (convergence speed depends on the singular value distribution of $G$; the more concentrated the singular values, the faster the convergence). Each iteration needs only a few matrix multiplications, an order of magnitude faster than SVD.

Muon's update equation (simplified):

$$\theta \leftarrow \theta - \eta \cdot \text{NewtonSchulz}(\text{momentum}(g))$$

where momentum is the usual exponential moving average (same as Adam's first moment). Muon's learning rate is typically an order of magnitude larger than Adam's ($4 \times 10^{-3}$ vs $3 \times 10^{-4}$), because the magnitude of the update direction is tightly controlled after orthogonalization and large steps are safe.

Muon is applied only to 2D matrix parameters. 1D parameters (bias, LayerNorm) have no matrix structure and orthogonalization is meaningless, so they still use Adam. In practice it is a hybrid optimizer: "Muon for 2D + Adam for 1D".


### 4.1 Intuitive verification of the Newton-Schulz iteration

Before writing the full Muon, verify in isolation that the Newton-Schulz iteration does orthogonalize a matrix. Take a random matrix $G$, iterate a few times, and check whether $X^\top X$ approaches the identity matrix.


In [ ]:
import torch

torch.manual_seed(42)


def newton_schulz(G, steps=7):
    """Approximately orthogonalize G using the Newton-Schulz iteration.

    Args:
        G: 2D gradient matrix
        steps: number of iterations; 7-10 is usually enough
    Returns:
        Approximately orthogonalized matrix with the same shape as G
    """
    assert G.dim() == 2, "Newton-Schulz only applies to 2D matrices"
    X = G.bfloat16() if G.is_cuda else G.float()
    # Normalize so the spectral radius < 1.5 to guarantee convergence
    X = X / (X.norm() + 1e-7)
    for _ in range(steps):
        A = X @ X.t()
        B = A @ X
        X = 1.5 * X - 0.5 * B
    return X


# Construct a random matrix and verify the orthogonalization effect of Newton-Schulz
G = torch.randn(8, 16)
X = newton_schulz(G, steps=7)

# Orthogonalization criterion: X X^T should be close to the identity
gram = X @ X.t()
identity = torch.eye(8)
off_diag = (gram - identity).abs().max().item()
diag_dev = (gram.diag() - 1.0).abs().max().item()

print(f"Input matrix G shape: {tuple(G.shape)}")
print(f"After 7 iterations, deviation of diag(X X^T) from 1: {diag_dev:.2e}")
print(f"After 7 iterations, max off-diagonal of X X^T:        {off_diag:.2e}")
print()
print("Key observation: after 7 Newton-Schulz iterations, X X^T is very close to the identity matrix.")
print("This is equivalent to replacing all singular values of G in its SVD with 1, preserving the direction while normalizing the magnitude.")


### 4.2 A simplified Muon optimizer

Embed Newton-Schulz inside `step()` and add momentum, and we have a minimal working Muon. The teaching version omits engineering details from the original such as RMS scaling and parameter-shape handling, but the core idea is intact.

The original Muon repo is at [KellerJordan/Muon](https://github.com/KellerJordan/Muon); in the nanoGPT Speedrun, Muon cut training time from 45 minutes to 22 minutes (modality=fp32, 50k steps).


In [ ]:
import torch


class SimpleMuon(torch.optim.Optimizer):
    """Simplified Muon: Newton-Schulz orthogonalization for 2D parameters, falls back to momentum SGD for 1D.

    Args:
        params: parameters to optimize
        lr: learning rate (Muon typically 4e-3, an order of magnitude larger than Adam)
        momentum: first moment decay coefficient
        ns_steps: number of Newton-Schulz iterations
    """

    def __init__(self, params, lr=4e-3, momentum=0.95, ns_steps=5):
        defaults = {"lr": lr, "momentum": momentum, "ns_steps": ns_steps}
        super().__init__(params, defaults)

    def step(self):
        for group in self.param_groups:
            lr = group["lr"]
            momentum = group["momentum"]
            ns_steps = group["ns_steps"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                grad = p.grad
                state = self.state[p]

                if len(state) == 0:
                    state["momentum_buffer"] = torch.zeros_like(p)

                buf = state["momentum_buffer"]
                buf.mul_(momentum).add_(grad)

                if p.dim() >= 2:
                    # 2D parameter: orthogonalize the momentum with Newton-Schulz
                    update = newton_schulz(buf, steps=ns_steps)
                    # Scale by sqrt(max(m, n)) to match the magnitude control of the original Muon
                    scale = max(1.0, buf.shape[0] / buf.shape[1]) ** 0.5
                    p.data.add_(update, alpha=-lr * scale)
                else:
                    # 1D parameter: fall back to plain momentum SGD
                    p.data.add_(buf, alpha=-lr)


# Simple smoke test: Muon runs and parameters change
torch.manual_seed(0)
p = torch.randn(32, 64, requires_grad=True)
p.grad = torch.randn(32, 64) * 0.1
before = p.data.clone()
opt = SimpleMuon([p], lr=4e-3, ns_steps=7)
opt.step()
delta = (p - before).abs().mean().item()
print(f"Mean parameter update magnitude: {delta:.6f}")
print(f"Parameter shape: {tuple(p.shape)}")
print("Key observation: Muon applies a Newton-Schulz-orthogonalized update direction to 2D parameters.")


## 5. The Lion / Shampoo / SOAP Lineage

Beyond Adam and Muon, 2023-2025 saw a few more optimizer routes worth knowing. Their shared motivation is that Adam treats each parameter position independently and ignores the structural information of the gradient as a matrix/tensor. Second-order information (correlations between gradients) is lost.

**Lion** (Google, 2023) has an extremely simple update rule:

$$\theta \leftarrow \theta - \eta \cdot \text{sign}(m)$$

where $m$ is the momentum. The difference from Adam is that Lion does not compute the second moment and directly takes the sign. The intuition is that the sign preserves the directional information of the gradient and discards the magnitude — this is equivalent to an extremely aggressive adaptive learning rate. Lion saves the storage of $v$, using one less copy of memory per parameter; it reports results on par with or better than AdamW on both Vision and LLMs. The cost is high sensitivity to the ratio between learning rate and weight decay, requiring re-tuning.

**Shampoo** (Gupta et al., 2018) is a true second-order optimizer. It maintains two preconditioning matrices $L = \mathbb{E}[g g^\top]$ (left) and $R = \mathbb{E}[g^\top g]$ (right), and uses $L^{-1/4} G R^{-1/4}$ as the update direction. This is the natural matrix generalization of Adam: Adam diagonalizes the covariance matrix, while Shampoo performs full-matrix preconditioning. The problem is that $L$ and $R$ are $m \times m$ and $n \times n$ matrices, making their storage and inversion prohibitively expensive, so Shampoo was long used only on small models.

**SOAP** (Shi et al., 2024) is the engineering-optimized version of Shampoo. It puts the updates of $L$ and $R$ on a low-frequency path (only every N steps) and reuses Adam's element-wise state for high-frequency correction. SOAP reports 30%-50% faster convergence than AdamW at LLaMA scale, bringing second-order optimizers back to large-model training.

**Why the second-order comeback in 2024-2026?** There are three reasons. First, growing hardware compute has made matrix preconditioning affordable. Second, exploding LLM training costs mean any convergence speedup translates to millions of dollars in savings. Third, hybrid schemes like Muon that "use matrix structure but do not go full second-order" prove that designs between Adam and Shampoo are viable.


## 6. In Practice: Adam vs Muon on miniGPT

With the theory covered, let's run a real comparison. Using the same MiniGPT architecture as in 06-mini-gpt (a small decoder-only LM), we train for 500 steps on a fixed text with AdamW and Muon respectively and observe the loss curves.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)


# Minimal MiniGPT, only for demonstrating optimizer comparison, with no engineering optimization
class MiniGPT(nn.Module):
    def __init__(self, vocab_size=256, d_model=64, n_heads=4, n_layers=2, max_len=64):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        # Simplified: stack one self-attention block n_layers times
        self.blocks = nn.ModuleList([
            nn.MultiheadAttention(d_model, n_heads, batch_first=True)
            for _ in range(n_layers)
        ])
        self.lns = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layers)])
        self.fc = nn.Linear(d_model, d_model * 4)
        self.proj = nn.Linear(d_model * 4, d_model)
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, idx):
        B, T = idx.shape
        pos = torch.arange(T)
        x = self.token_emb(idx) + self.pos_emb(pos)
        for block, ln in zip(self.blocks, self.lns):
            h = ln(x)
            # causal self-attention
            mask = torch.triu(torch.full((T, T), float("-inf")), diagonal=1)
            attn_out, _ = block(h, h, h, attn_mask=mask)
            x = x + attn_out
            x = x + self.proj(F.gelu(self.fc(x)))
        x = self.ln_f(x)
        return self.lm_head(x)


# Prepare a fixed text as training corpus (Shakespeare snippet, byte-level encoding)
text = ("to be or not to be that is the question "
        "whether tis nobler in the mind to suffer "
        "the slings and arrows of outrageous fortune "
        "or to take arms against a sea of troubles ")
data = torch.tensor([ord(c) for c in text], dtype=torch.long)
print(f"Corpus length: {len(data)} bytes")
print(f"Vocab size: 256 (byte-level)")
print(f"First 40 bytes: {data[:40].tolist()}")


In [ ]:
import torch
import torch.nn.functional as F


def make_optimizer(model, opt_name):
    """Build different optimizers for the same model."""
    if opt_name == "AdamW":
        return torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
    elif opt_name == "Muon":
        # Muon recipe: Muon for 2D, Adam for 1D
        decay_2d = [p for p in model.parameters() if p.dim() >= 2]
        decay_1d = [p for p in model.parameters() if p.dim() < 2]
        return SimpleMuon([
            {"params": decay_2d, "lr": 4e-3},
            {"params": decay_1d, "lr": 3e-4},
        ])


def train_and_record(opt_name, steps=500, seq_len=32, batch=8):
    """Train a model and record the loss curve."""
    torch.manual_seed(42)
    model = MiniGPT()
    opt = make_optimizer(model, opt_name)

    losses = []
    for step in range(steps):
        # Randomly sample a batch
        starts = torch.randint(0, len(data) - seq_len - 1, (batch,))
        idx = torch.stack([data[s:s + seq_len] for s in starts])
        target = torch.stack([data[s + 1:s + 1 + seq_len] for s in starts])

        logits = model(idx)
        loss = F.cross_entropy(logits.reshape(-1, 256), target.reshape(-1))

        opt.zero_grad()
        loss.backward()
        # All optimizers use gradient clipping (max_norm=1.0) for a fair comparison
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        losses.append(loss.item())

    return losses


# Run two experiments
print("Starting training, 500 steps each...")
losses_adamw = train_and_record("AdamW")
losses_muon = train_and_record("Muon")
print("Training done.")
print()
print(f"AdamW  initial loss: {losses_adamw[0]:.3f}, after 500 steps: {losses_adamw[-1]:.3f}")
print(f"Muon   initial loss: {losses_muon[0]:.3f}, after 500 steps: {losses_muon[-1]:.3f}")


In [ ]:
import matplotlib.pyplot as plt


def smooth(xs, window=20):
    if len(xs) < window:
        return xs
    return [sum(xs[max(0, i - window):i + 1]) / min(i + 1, window) for i in range(len(xs))]


plt.figure(figsize=(9, 4))
plt.plot(smooth(losses_adamw), label="AdamW (lr=3e-4)", alpha=0.85)
plt.plot(smooth(losses_muon), label="Muon (lr=4e-3)", alpha=0.85)
plt.xlabel("step")
plt.ylabel("loss (smoothed, window=20)")
plt.title("AdamW vs Muon on miniGPT (500 steps, byte-level Shakespeare)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Key observations:")
print(f"- AdamW takes loss from {losses_adamw[0]:.2f} to {losses_adamw[-1]:.2f} within 500 steps")
print(f"- Muon  takes loss from {losses_muon[0]:.2f} to {losses_muon[-1]:.2f} within 500 steps")
print("- Muon's learning rate is an order of magnitude larger than AdamW's, yet training remains stable — orthogonalization keeps the step size controllable.")
print("- On larger datasets and larger models, Muon's advantage is more pronounced (see the nanoGPT Speedrun).")


### 6.1 Learning rate sensitivity

Muon's learning rate is an order of magnitude larger than AdamW's, and this is not a coincidence. Adam's update magnitude is compressed by $1/\sqrt{v}$, while Muon's update has its magnitude normalized to a stable range by orthogonalization, so large steps are safe. Running Muon directly with Adam's learning rate ($3 \times 10^{-4}$) would update too slowly; conversely, running AdamW with Muon's learning rate ($4 \times 10^{-3}$) would diverge immediately on a real large model (the small model below has gradient clipping as a safety net, so the divergence is not obvious).

Let's quickly verify this by running a few steps with different learning rates and comparing the sensitivity of the two optimizers.


In [ ]:
import torch
import torch.nn.functional as F


def quick_run(opt_name, lr, steps=50):
    """Quickly run 50 steps and return the final loss."""
    torch.manual_seed(42)
    model = MiniGPT()
    if opt_name == "AdamW":
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    else:
        decay_2d = [p for p in model.parameters() if p.dim() >= 2]
        decay_1d = [p for p in model.parameters() if p.dim() < 2]
        opt = SimpleMuon([
            {"params": decay_2d, "lr": lr},
            {"params": decay_1d, "lr": lr / 10},
        ])

    last_loss = None
    for step in range(steps):
        starts = torch.randint(0, len(data) - 33, (8,))
        idx = torch.stack([data[s:s + 32] for s in starts])
        target = torch.stack([data[s + 1:s + 33] for s in starts])
        logits = model(idx)
        loss = F.cross_entropy(logits.reshape(-1, 256), target.reshape(-1))
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        last_loss = loss.item()
    return last_loss


print("Learning rate sensitivity comparison (loss after 50 steps, lower is better):")
print(f"{'Optimizer':<10} {'lr=3e-4':<14} {'lr=1e-3':<14} {'lr=4e-3':<14}")
print("-" * 56)
for name in ["AdamW", "Muon"]:
    row = f"{name:<10} "
    for lr in [3e-4, 1e-3, 4e-3]:
        loss = quick_run(name, lr)
        row += f"{loss:<14.3f}"
    print(row)

print()
print("Key observations:")
print("- AdamW performs best at lr=3e-4; on this toy model a larger lr actually converges faster (because clipping is on).")
print("- Muon barely updates at lr=3e-4 (loss still around 5.5) and needs lr=4e-3 to work properly.")
print("- Core takeaway: Muon's effective learning rate is roughly an order of magnitude larger than AdamW's; you cannot reuse Adam's lr directly.")
print("- On a real large model (no clip or weaker clip), AdamW with Muon's lr would diverge immediately.")


## 7. A Selection Decision Tree

With so many optimizers covered, which one should you choose in a real project? This section gives an engineering decision framework. There are three core dimensions: training stage (pretraining / SFT / RLHF), model scale (small / medium / large), and engineering constraints (memory / time / stability).

**Pretraining**: AdamW is still the first choice, with the community most thoroughly tuned on its hyperparameters ($\beta_1=0.9, \beta_2=0.95, \text{wd}=0.1, \text{lr}=3\times 10^{-4}$). Muon proved its speed advantage in the 2024 nanoGPT Speedrun, but data on stability at the hundred-billion-parameter scale is still insufficient; it is currently used mainly in medium-scale (hundreds of millions to a few billion) experiments. SOAP is a potential replacement for AdamW and has been validated at LLaMA scale, but the community toolchain is not yet mature.

**SFT / instruction tuning**: AdamW remains the default. SFT batches are usually small and the number of steps is low (thousands), so the optimizer's convergence speed advantage is not pronounced and stability matters more. Learning rate drops to $10^{-5}$ to $10^{-4}$, and weight decay is usually 0 or very small.

**RLHF (PPO stage)**: AdamW, but note that the actor and critic use different learning rates (the actor's is usually smaller). The critic's loss is MSE, with gradients of a completely different scale from the LM loss, requiring separate clipping.

**Inference**: no optimizer is involved. Just a reminder: at inference time all optimizer state can be discarded, saving memory.

A common engineering pitfall is "wanting to switch whenever a new optimizer appears". AdamW has been validated across dozens of hundred-billion models including GPT-3, LLaMA, Qwen, and DeepSeek, and its hyperparameter space is well explored. Switching to Muon or SOAP requires re-tuning the learning rate, weight decay, and warmup, which is not cheap. Unless there is a clear throughput bottleneck, AdamW with a good LR schedule is the safest choice.


## 8. Optimizer Considerations for MoE

Mixture-of-Experts (MoE) models present two unique challenges at the optimizer level: the expert parameters are large (VRAM usage explodes) and the router parameters have gradient stability concerns. 10-moe covered the structure of the MoE layer; here we expand from the optimizer's perspective.

**VRAM pressure from expert parameters**. An 8-expert MoE layer has 8x the FFN parameters of a dense model. Adam needs to maintain $m$ and $v$ for each parameter, so the optimizer state is also 8x. This means the MoE model that fits on a given GPU is much smaller than the dense model. Two coping strategies: first, 8-bit Adam (quantize $m$ and $v$ to 8-bit, saving 75% of memory); second, optimizer-state sharding such as ZeRO-3 (shard $m$ and $v$ across cards).

**Stability of router parameters**. The router (gate) is a small linear layer whose softmax output decides which expert each token goes to. Its gradient has two sources: the backprop from the normal LM loss and the auxiliary loss for load balancing. If the router gradient updates too fast, a collapse can occur — "a few experts suddenly get selected heavily while others sit completely idle" — and once it happens it is hard to recover. Common engineering fixes: give the router a separate, smaller learning rate (e.g. 0.1x the rest), or apply extra dropout to the router output.

**DeepSeek-V3's optimizer configuration**. The DeepSeek-V3 technical report explicitly states it uses AdamW with $\beta_1=0.9, \beta_2=0.95$, and weight decay set to 0.1 and 0 on different parameter groups. The router parameters go in an independent parameter group. The learning rate schedule is multi-step decay with warmup. This is a typical modern MoE training recipe, differing from a dense model mainly in parameter grouping and learning rate ratios, not in the optimizer algorithm itself.

**Mixtral / Kimi K2 choices**. Mixtral still uses AdamW (the community generally reproduces the LLaMA training stack). Kimi K2 (Moonshot) made many optimizer-side engineering changes for MoE, but the core algorithm is still the AdamW family. Muon on MoE is still in the research stage — expert parameters are 2D matrices and in principle suit Muon, but the router's small size makes Muon's matrix orthogonalization benefit marginal, so it still uses Adam.

**Gradient update stability for expert routing** boils down to a few rules of thumb: router learning rate < expert learning rate; apply stronger weight decay or dropout to router parameters; monitor token traffic per expert and alert when it exceeds a threshold (e.g. 3x uniform); and the weight of the auxiliary loss needs gradual adjustment (Switch Transformer uses 0.01; DeepSeek uses a more complex form).


### 8.1 Optimizer configuration: dense vs MoE

The table below summarizes the differences between the two model types at the optimizer level for easy comparison.

| Dimension | Dense model (LLaMA / Qwen) | MoE model (Mixtral / DeepSeek-V3) |
|:---|:---|:---|
| Optimizer | AdamW | AdamW (sometimes paired with 8-bit Adam on experts to save memory) |
| Parameter groups | decay / no_decay (two groups) | decay / no_decay / router (at least three) |
| Router learning rate | N/A | usually < expert lr (0.1-0.5x) |
| Weight decay | 0.1 (Linear/Embedding), 0 (bias/LN) | same as left, router tuned separately |
| Gradient clip | max_norm=1.0 | same as left, but clipping router and expert separately is more stable |
| Auxiliary loss | none | load balancing loss, weight on the order of 0.01 |
| Optimizer state memory | 2x parameters (fp32 m and v) | 8x the dense expert parameters, use ZeRO-3 / 8-bit |
| Alternative optimizers | Muon, SOAP have experimental data | Muon has potential on expert 2D parameters; router still uses Adam |


## Summary

Confirm you understand the following points:

- [ ] Adam maintains the first moment $m$ (momentum) and second moment $v$ (variance), with update $\theta \leftarrow \theta - \eta \hat{m} / (\sqrt{\hat{v}} + \epsilon)$
- [ ] bias correction fixes initialization bias: $\hat{m}_t = m_t / (1 - \beta_1^t)$, so early updates are not underestimated
- [ ] AdamW decouples weight decay from the gradient chain, directly via $\theta \leftarrow \theta - \eta \lambda \theta$, avoiding being scaled by the adaptive learning rate
- [ ] bias and LayerNorm parameters get no weight decay, distinguished via parameter groups
- [ ] gradient clipping uses global norm clip ($\tau = 1.0$), preserving direction and only scaling magnitude
- [ ] Muon uses the Newton-Schulz iteration to orthogonalize the gradient matrix, preserving direction and normalizing magnitude, with a learning rate an order of magnitude larger than Adam
- [ ] Muon is applied only to 2D matrix parameters; 1D parameters (bias / LayerNorm) fall back to momentum SGD or Adam
- [ ] Lion takes the sign of the momentum, saving memory but is sensitive to hyperparameters; Shampoo / SOAP are second-order optimizers that came back in 2024 under training-cost pressure
- [ ] Switching optimizers requires re-tuning the learning rate; Adam's lr cannot be used directly with Muon
- [ ] The router parameters of an MoE model need their own parameter group, with a learning rate usually smaller than the expert parameters
- [ ] AdamW is still the de facto standard for large-model pretraining; the advantages of newer optimizers (Muon / SOAP) mainly show up in medium-scale experiments


## Exercises

> You may ask AI to explain ideas, break down steps, and check directions, but it is not recommended to have AI "complete this problem for you".

**Exercise 1: Hand-calculate one Adam step**

Given parameter $\theta_0 = 2.0$, $m_0 = v_0 = 0$, hyperparameters $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\eta = 0.01$, $\epsilon = 10^{-8}$, and first-step gradient $g_1 = -1.0$. Hand-calculate $\theta_1$.

Hint: first compute $m_1$ and $v_1$, then apply bias correction, then plug into the update equation.


In [ ]:
import torch

# TODO: hand-compute theta_1
manual_theta1 = None  # fill in your hand-computed result here

# Verify with PyTorch
theta = torch.tensor([2.0], requires_grad=True)
theta.grad = torch.tensor([-1.0])
opt = torch.optim.Adam([theta], lr=0.01, betas=(0.9, 0.999), eps=1e-8)
opt.step()
torch_theta1 = theta.item()

assert manual_theta1 is not None, "Please hand-compute manual_theta1 first"
assert abs(manual_theta1 - torch_theta1) < 1e-5, \
    f"Answer should be {torch_theta1:.6f}, you got {manual_theta1}"

print("✅ Exercise 1 passed:")
print(f"   theta_0 = 2.0, g_1 = -1.0")
print(f"   theta_1 = {manual_theta1:.6f}")
print(f"   After bias correction the first-step gradient restores to g_1 = -1.0, update magnitude = lr * 1 = 0.01")


**Exercise 2: Implement global norm clip**

Implement a `clip_global_norm` function that takes a parameter list as input, returns the pre-clip global norm, and proportionally scales down any gradient exceeding `max_norm`. You may not call `torch.nn.utils.clip_grad_norm_`.

Hint: global norm $= \sqrt{\sum_i \|g_i\|^2}$, clip coefficient $= \min(1, \tau / \|g\|)$.


In [ ]:
import torch


def clip_global_norm(params, max_norm):
    """Hand-written global norm clip.

    Args:
        params: list of parameters (some grads may be None)
        max_norm: upper bound on the global norm
    Returns:
        the pre-clip global norm (a scalar)
    """
    # TODO: implement
    grads = [p.grad for p in params if p.grad is not None]
    total_norm = None  # square root of the sum of squared L2 norms of all gradients
    clip_coef = None   # min(1, max_norm / (total_norm + eps))
    # If clip_coef < 1, in-place scale all gradients
    return total_norm


# Test
torch.manual_seed(0)
params_a = [torch.randn(50, requires_grad=True) for _ in range(3)]
params_b = [p.clone().detach().requires_grad_(True) for p in params_a]
for pa, pb in zip(params_a, params_b):
    pa.grad = torch.randn(50) * 5
    pb.grad = pa.grad.clone()

# Our method
norm_a = clip_global_norm(params_a, max_norm=1.0)
# PyTorch's method
norm_b = torch.nn.utils.clip_grad_norm_(params_b, max_norm=1.0).item()

after_a = torch.sqrt(sum(p.grad.float().pow(2).sum() for p in params_a)).item()
after_b = torch.sqrt(sum(p.grad.float().pow(2).sum() for p in params_b)).item()

assert norm_a is not None, "Please implement clip_global_norm first"
assert abs(norm_a - norm_b) < 1e-4, f"global norm should be {norm_b:.4f}, you got {norm_a}"
assert abs(after_a - after_b) < 1e-4, f"post-clip norm should be {after_b:.4f}, you got {after_a}"

print(f"✅ Exercise 2 passed:")
print(f"   global norm before clip: {norm_a:.4f}")
print(f"   global norm after clip:  {after_a:.4f}")
print(f"   Matches the PyTorch implementation exactly")


**Exercise 3: Implement a Muon + Adam hybrid optimizer**

Implement a `MuonAdam` class: apply Newton-Schulz orthogonalization to 2D parameters (call `newton_schulz` defined earlier in this section), and Adam to 1D parameters. This is the standard Muon recipe in production.

Hint: in `__init__`, split the parameters into two groups by dimension and store them in `self.muon_params` and `self.adam_params`. In `step()`, update the two groups separately.


In [ ]:
import torch


class MuonAdam:
    """Muon + Adam hybrid optimizer.

    2D parameters use Muon (Newton-Schulz orthogonalization); 1D parameters use Adam.
    """

    def __init__(self, params, muon_lr=4e-3, adam_lr=3e-4,
                 momentum=0.95, betas=(0.9, 0.999), eps=1e-8, ns_steps=5):
        params = list(params)
        # TODO: group by dimension
        self.muon_params = None  # 2D parameters
        self.adam_params = None  # 1D parameters
        self.muon_lr = muon_lr
        self.momentum = momentum
        self.ns_steps = ns_steps
        # Use torch.optim.Adam for the 1D parameters
        self.adam_opt = None
        self.state = {}

    def zero_grad(self):
        for p in self.muon_params + self.adam_params:
            if p.grad is not None:
                p.grad.zero_()

    def step(self):
        # TODO: use Newton-Schulz to update muon_params
        # call self.adam_opt.step() for adam_params
        pass


# Test: build a small model and verify MuonAdam updates normally
torch.manual_seed(0)
model = torch.nn.Sequential(
    torch.nn.Linear(32, 64),
    torch.nn.LayerNorm(64),
    torch.nn.Linear(64, 32),
)

# Build a dummy loss and backward
x = torch.randn(4, 32)
target = torch.randn(4, 32)
loss = ((model(x) - target) ** 2).mean()
loss.backward()

# Record parameter values before the update
before = [p.data.clone() for p in model.parameters()]

opt = MuonAdam(model.parameters())
opt.step()

# Verify that parameters actually changed
changed = any(
    not torch.equal(b, p.data)
    for b, p in zip(before, model.parameters())
)

assert opt.muon_params is not None and opt.adam_params is not None, \
    "Please implement the grouping in __init__ first"
assert changed, "Parameters did not change; something is wrong with step()"

n_muon = len(opt.muon_params)
n_adam = len(opt.adam_params)
print(f"✅ Exercise 3 passed:")
print(f"   Muon param count (2D): {n_muon}")
print(f"   Adam param count (1D): {n_adam}")
print(f"   All parameters were updated; grouping is correct.")
print(f"   This is the standard Muon recipe from the nanoGPT Speedrun.")
